In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.llms.litellm import LiteLLM
from src.food_rag.utils import embedding_helper, hummus_docify
from src.utils import load_util
from src.persona import qa_generator
from src.food_rag.utils.embedding_helper import QA_SUB_PATH

In [ ]:
# Output Path
data_path = './data'
embedding_path = data_path + '/embeddings/snowflake_l'
qa_name = 'small_1'

## Raw HUMMUS data
data_location = '....'
graph_location = '....'
additional_location = '....'

# Load configs
nutrition_config = load_util.get_config('config/nutrition_config.yml')
persona_config = load_util.get_config('config/persona_config.yml')

# Load key
api_key = load_util.load_file('config/api_key')

In [ ]:
# Init LLM
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1",
    max_retries=100,
    cooldown_time=60,
)

# Remote Embedding model
embed_model = LiteLLMEmbedding(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model_name="openai/Snowflake/snowflake-arctic-embed-l",
)

# Local Embedding model # remote using nvidia/NV-Embed-v2
# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en", trust_remote_code=True, cache_folder=data_path + '/cache')
#for module_key, module in embed_model._modules.items():
#    embed_model._modules[module_key] = DataParallel(module)

# Set Global Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

In [ ]:
# Load (init if necessary) the data; NOTE: create random meal types annotations for the test set
hummus_data = load_util.get_hummus(data_location, graph_location, additional_location, data_path=data_path,
                                   discard_non_tagged_recipes=False, n_clusters=-1, k_user=25, k_recipe=25,
                                   food_class_strategy='None', add_additional_meal_type_labels=True,
                                   clean_food_category=True, similarity_thresholds=False, include_usda=True,
                                   do_annotate_meal_type=True, create_random_meal_types=True, llm=llm)

# Load the qa data
question_answer_sets = qa_generator.load_qa_set(data_path, sub_dir_name=qa_name, include_full_recipes=False)

In [ ]:
# Get an overview of the data and avg context length
tokenizer = embedding_helper.get_tokenizer_from_embed_model(embed_model.model_name.removeprefix("openai/"))
lengths = [len(tokenizer.tokenize(doc.text)) for doc in hummus_docify.recipes_to_docs(hummus_data['recipes_df'][:100])]
avg_length = sum(lengths) / len(lengths)
print(f'Average context length: {avg_length}')

# Embedd documents

In [ ]:
%%capture_output_and_exceptions capture_recipe_documents
# Embedd Recipes
recipe_documents = hummus_docify.recipes_to_docs(hummus_data['recipes_df'])
recipe_doc_dict = embedding_helper.compute_embeddings(recipe_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + '/recipe_documents')

In [ ]:
%%capture_output_and_exceptions capture_user_documents
# Embedd Users
user_documents = hummus_docify.users_to_docs(hummus_data['users_df'])
user_doc_dict = embedding_helper.compute_embeddings(user_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + '/user_documents')

In [ ]:
%%capture_output_and_exceptions capture_review_documents
# Embedd Reviews. The returned result will not be complete due to the large size of the dataset. For that it is required to load the embeddings later again.
review_documents = hummus_docify.reviews_to_docs(hummus_data['reviews_df'])
review_doc_dict = embedding_helper.compute_embeddings(review_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + '/review_documents', minimize_memory=True)

In [ ]:
%%capture_output_and_exceptions capture_ingredient_documents
# Embedd Ingredients
ingredient_documents = hummus_docify.ingredients_to_docs(hummus_data['ingredients_df'])
ingredient_doc_dict = embedding_helper.compute_embeddings(ingredient_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + '/ingredient_documents')

In [ ]:
%%capture_output_and_exceptions capture_food_product_documents
# Embedd Food Products. The returned result will not be complete due to the large size of the dataset. For that it is required to load the embeddings later again.
_ = embedding_helper.compute_embeddings(hummus_docify.food_products_to_docs(hummus_data['usda_df']), embed_model, batch_size=1000, chunk_size=300, path=embedding_path + '/food_product_documents', minimize_memory=True)

In [ ]:
%%capture_output_and_exceptions capture_food_on_documents
# Embedd Food On
food_on_documents = hummus_docify.food_on_to_docs(hummus_data['food_on_ontology'])
food_on_doc_dict = embedding_helper.compute_embeddings(food_on_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + '/food_on_documents')

# Embedd questions from the qa set

In [ ]:
%%capture_output_and_exceptions capture_qa_meal_plan_documents
# Embedd Questions
if 'meal_plan_question_answer_set' in question_answer_sets:
    qa_meal_plan_documents = hummus_docify.qa_question_to_docs(question_answer_sets['meal_plan_question_answer_set'])
    qa_meal_plan_doc_dict = embedding_helper.compute_embeddings(qa_meal_plan_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + QA_SUB_PATH + qa_name + '/qa_meal_plan_documents', embedd_query=True)

In [ ]:
%%capture_output_and_exceptions capture_qa_recipe_documents
if 'recipe_question_answer_set' in question_answer_sets:
    qa_recipe_documents = hummus_docify.qa_question_to_docs(question_answer_sets['recipe_question_answer_set'])
    qa_recipe_doc_dict = embedding_helper.compute_embeddings(qa_recipe_documents, embed_model, batch_size=1000, chunk_size=300, path=embedding_path + QA_SUB_PATH + qa_name + '/qa_recipe_documents', embedd_query=True)

# Load embeddings

In [ ]:
recipe_loaded_doc_dict, recipe_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/recipe_documents')
user_loaded_doc_dict, user_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/user_documents')
review_loaded_doc_dict, review_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/review_documents')
ingredient_loaded_doc_dict, ingredient_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/ingredient_documents')
food_loaded_product_doc_dict, food_product_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/food_product_documents')
food_loaded_on_doc_dict, food_on_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + '/food_on_documents')

In [ ]:
qa_meal_plan_loaded_doc_dict, qa_meal_plan_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + QA_SUB_PATH + qa_name + '/qa_meal_plan_documents')
qa_recipe_loaded_doc_dict, qa_recipe_loaded_batch_names = embedding_helper.load_existing_batches(embedding_path + QA_SUB_PATH + qa_name + '/qa_recipe_documents')